In [ ]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

In [ ]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=3:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=4)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

2025-11-24 14:12:24,258 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2025-11-24 14:12:24,260 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2025-11-24 14:12:24,261 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2025-11-24 14:12:24,262 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB


In [4]:
DATA_ROOT="/home/mcn26/project_pi_skr2/shared/tabula_data"
simu_obj=scm.de_novo_simulation.load(client,path=f"{DATA_ROOT}/simulated/shendure_pow_analysis",name="sim_20251119")

In [ ]:
#temporary : obj created w/ older version of code, necessitating this. 
simu_obj.orthos=[]

In [ ]:
simu_obj.create_orthos_for_all_replicates(client)

In [ ]:
simu_obj.save(path=f"{DATA_ROOT}/simulated/shendure_pow_analysis",name="sim_with_orthos_20251124")

In [6]:
client.close()
cluster.close()